In [11]:
import requests

In [12]:
def fetch_ids():
    url = (
        "http://searchsolrv2-mirror-prod.us-east1.tgt-pe-prod.gcp.cloud.target.internal:8983"
        "/solr/catalog/searchv2?q=*%3A*&fl=id&rows=28566993"
    )

    resp = requests.get(url, timeout=300)
    resp.raise_for_status()  # raise error if request failed
    
    data = resp.json()

    docs = data.get("response", {}).get("docs", [])
    print(f"Fetched {len(docs)} documents from Solr.")

    result_ids = []
    
    for d in docs:
        raw_id = d.get("id", "")
        if "!" in raw_id:
            after = raw_id.split("!", 1)[1]
            result_ids.append(after)
        else:
            result_ids.append(raw_id)

    return result_ids

In [13]:
def write_ids_to_txt(ids, filename="active_tcins.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for item in ids:
            f.write(item + "\n")
    print(f"Wrote {len(ids)} IDs to {filename}")


In [17]:
tcins = fetch_ids()
print(f"Collected {len(tcins)} ids")
print(tcins[:10])

Fetched 6820503 documents from Solr.
Collected 6820503 ids
['1006968493', '1006066115', '1001546656', '1006562642', '1006619539', '1005279134', '1003027298', '1006390779', '1006397257', '1006521503']


In [ ]:
write_ids_to_txt(tcins, "active_tcins.txt")

Wrote 6884964 IDs to active_tcins.txt


# Write tcin with title

In [ ]:
import requests
import time
import csv

SOLR_BASE = "http://searchsolrv2-mirror-prod.us-east1.tgt-pe-prod.gcp.cloud.target.internal:8983/solr/catalog/searchv2"
OUT_CSV = "tcins_with_titles.csv"

def fetch_tcins_with_titles(batch_size=10000, max_retries=3, sleep_on_retry=5):
    params = {
        "q": "*:*",
        "fl": "id,p_title",
        "rows": batch_size,
        "sort": "id asc", 
        "cursorMark": "*"
    }

    next_cursor = params["cursorMark"]
    total_fetched = 0

    with open(OUT_CSV, "w", encoding="utf-8", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["tcin", "p_title"])

        while True:
            params["cursorMark"] = next_cursor
            for attempt in range(1, max_retries+1):
                try:
                    resp = requests.get(SOLR_BASE, params=params, timeout=(10, 120))
                    resp.raise_for_status()
                    data = resp.json()
                    break
                except Exception as e:
                    print(f"Request error (attempt {attempt}/{max_retries}): {e}")
                    if attempt == max_retries:
                        raise
                    time.sleep(sleep_on_retry)

            docs = data.get("response", {}).get("docs", [])
            if not docs:
                print("No more docs returned, finishing.")
                break

            for d in docs:
                raw_id = d.get("id", "")
                # safety: get title (may be absent)
                title = d.get("p_title", "") or ""
                if "!" in raw_id:
                    after = raw_id.split("!", 1)[1]
                    writer.writerow([after, title])
                    total_fetched += 1
                else:
                    writer.writerow([raw_id, title])
                    total_fetched += 1

            next_cursor = data.get("nextCursorMark")
            print(f"Fetched batch, docs={len(docs)}, total={total_fetched}, nextCursor={next_cursor}")

            if not next_cursor:
                print("No nextCursorMark returned, stopping.")
                break
            if next_cursor == params["cursorMark"]:
                print("cursorMark did not advance, stopping to avoid infinite loop.")
                break

    print(f"Done. Total extracted id/title pairs: {total_fetched}")


In [ ]:
fetch_tcins_with_titles(batch_size=100000)

# Read and Check

In [ ]:
import pandas as pd

df = pd.read_csv("tcins_with_titles.csv")

df_filtered = df.dropna(subset=["p_title"])

df_filtered.to_csv("tcins_with_titles.csv", index=False)

print("Original rows:", len(df))
print("Filtered rows:", len(df_filtered))
